# ADIM 6 — Koşullu Transfer Probe (barajlı, dürüst)

Bir sette eğitilen modelin BAŞKA sette ne kadar genellediğini ölçer. İki tip:
**(a) sektör-içi** (Cell2Cell↔Iranian, iki yön) — çalışması beklenir;
**(b) sektör-ötesi** (telco↔bank, iki yön) — çökmesi beklenir.
Setler birleştirilmez; transfer = "kaynakta fit, hedefte predict". Ortak uzay =
concept_map kavramlarından her iki sette de SAYISAL temsilcisi olanlar (kavram
başına tek temsilci). Scaler yalnız kaynakta fit (sızıntı yok). Model: HAM LightGBM.
Sonuç **önceden tanımlı barajla** üç kategoriye atanır (DAHİL/KISMÎ/ZAYIF) — sonuca
göre eğilmez; başarısızlık da bir bulgudur. Ağır mantık `src/transfer.py`'de.

In [1]:
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd


def _bul_kok():
    for c in [Path.cwd(), *Path.cwd().parents]:
        if (c / "config.yaml").exists():
            return c
    raise RuntimeError("config.yaml bulunamadı")


KOK = _bul_kok()
if str(KOK) not in sys.path:
    sys.path.insert(0, str(KOK))

warnings.filterwarnings("ignore")
from src import config as cfg
from src import plotstyle as ps
from src import strings_tr as S
from src import transfer as tr

np.random.seed(cfg.SEED)
ps.uygula()
cfg.klasorleri_hazirla()
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

CIKTI = []


def yaz(s=""):
    print(s)
    CIKTI.append(str(s))

## 1. Veri yükleme

In [2]:
veriler = {k: pd.read_csv(cfg.PROCESSED / f"{k}_clean.csv") for k in cfg.DATASETS}
for k, d in veriler.items():
    yaz(f"{k:11s} {d.shape}")

telco       (7043, 20)
cell2cell   (51047, 57)
ecommerce   (3941, 11)
iranian     (3150, 14)
bank        (10000, 11)


## 2. Dört senaryo: ortak-kavram uzayında transfer + in-domain referans + trivial
Ortak-kavram sayısı sektör-ötesinde muhtemelen daha az olacak — transferin neden
zayıf olduğunun kanıtı.

In [3]:
yaz(S.MSG["bolum"].format(ad="TRANSFER PROBE"))
sonuclar = []
for anahtar, ad, ks, ht, tip in tr.SENARYOLAR:
    t = time.time()
    r = tr.calistir_senaryo(anahtar, ad, ks, ht, tip, veriler, cfg.SEED)
    sonuclar.append(r)
    yaz(S.MSG6["senaryo"].format(ad=r["ad"], yon=r["tip"], k=len(r["ortak"]),
        tr=r["transfer"], ref=r["ref"], tv=r["trivial"], oran=r["oran"],
        karar=S.TRANSFER_KARAR[r["karar"]]))
    yaz(f"    ortak kavram eşlemesi: {[r['esleme'][k] for k in r['ortak']]}  ({time.time()-t:.0f}s)")

===== TRANSFER PROBE =====


Cell2Cell -> Iranian (sektör-içi): ortak kavram=5 | transfer PR-AUC=0.153 | ref=0.845 | trivial=0.157 | oran=0.18 -> ZAYIF
    ortak kavram eşlemesi: [('MonthsInService', 'Subscription  Length'), ('MonthlyMinutes', 'Status'), ('DroppedCalls', 'Complains'), ('TotalRecurringCharge', 'Customer Value'), ('AgeHH1', 'Age Group')]  (18s)


Iranian -> Cell2Cell (sektör-içi): ortak kavram=5 | transfer PR-AUC=0.278 | ref=0.404 | trivial=0.288 | oran=0.69 -> ZAYIF
    ortak kavram eşlemesi: [('Subscription  Length', 'MonthsInService'), ('Status', 'MonthlyMinutes'), ('Complains', 'DroppedCalls'), ('Customer Value', 'TotalRecurringCharge'), ('Age Group', 'AgeHH1')]  (15s)


telco -> bank (sektör-ötesi): ortak kavram=3 | transfer PR-AUC=0.237 | ref=0.450 | trivial=0.204 | oran=0.53 -> KISMÎ
    ortak kavram eşlemesi: [('tenure', 'Tenure'), ('MonthlyCharges', 'Balance'), ('SeniorCitizen', 'Age')]  (4s)


bank -> telco (sektör-ötesi): ortak kavram=3 | transfer PR-AUC=0.207 | ref=0.622 | trivial=0.265 | oran=0.33 -> ZAYIF
    ortak kavram eşlemesi: [('Tenure', 'tenure'), ('Balance', 'MonthlyCharges'), ('Age', 'SeniorCitizen')]  (4s)


## 3. Tablolar
`transfer_results.csv` (senaryo, ortak kavram, transfer/ref/trivial PR-AUC, koruma
oranı, recall/precision/F1, karar) ve `transfer_feature_map.csv` (kavram eşlemesi).

In [4]:
sonuc_df = tr.tablo_sonuc(sonuclar)
map_df = tr.tablo_feature_map(sonuclar)
yaz(S.MSG["bolum"].format(ad="TRANSFER SONUÇLARI"))
yaz(sonuc_df.to_string(index=False))
yaz(S.MSG["kayit"].format(yol=cfg.TABLES / "transfer_results.csv"))
yaz(S.MSG["kayit"].format(yol=cfg.TABLES / "transfer_feature_map.csv"))

===== TRANSFER SONUÇLARI =====
Senaryo                  Yön  Ortak kavram sayısı  Transfer PR-AUC  In-domain ref PR-AUC  Tam-feature ref PR-AUC  Trivial PR-AUC  Koruma oranı  Duyarlılık  Kesinlik     F1 Karar
     A1 Cell2Cell -> Iranian                    5           0.1531                0.8449                  0.9576          0.1571         0.181      0.4242    0.1308 0.2000 ZAYIF
     A2 Iranian -> Cell2Cell                    5           0.2783                0.4035                  0.4661          0.2882         0.690      0.8214    0.2829 0.4209 ZAYIF
     B1        telco -> bank                    3           0.2373                0.4503                  0.7065          0.2037         0.527      0.7545    0.2408 0.3651 KISMÎ
     B2        bank -> telco                    3           0.2075                0.6219                  0.6635          0.2654         0.334      0.0000    0.0000 0.0000 ZAYIF
Kaydedildi: /Users/emrahfidan/Desktop/churn-xai-profit/outputs/tables/transfer_

## 4. Figürler

In [5]:
yaz(S.MSG["bolum"].format(ad="FİGÜRLER"))
y1 = tr.figur_prauc(sonuclar)
y2 = tr.figur_retention(sonuclar)
yaz(S.MSG["kayit"].format(yol=y1))
yaz(S.MSG["kayit"].format(yol=y2))

===== FİGÜRLER =====
Kaydedildi: /Users/emrahfidan/Desktop/churn-xai-profit/outputs/figures/_transfer/transfer_prauc_comparison.png
Kaydedildi: /Users/emrahfidan/Desktop/churn-xai-profit/outputs/figures/_transfer/transfer_retention_ratio.png


## 5. Özet — sektör-içi gerçekten daha mı iyi? (karar kullanıcıda)

In [6]:
yaz(S.MSG["bolum"].format(ad="ÖZET — TRANSFER"))
ici = [r for r in sonuclar if r["tip"] == "sektör-içi"]
otesi = [r for r in sonuclar if r["tip"] == "sektör-ötesi"]
ici_ort = np.mean([r["oran"] for r in ici])
otesi_ort = np.mean([r["oran"] for r in otesi])
for r in sonuclar:
    yaz(f"  {r['anahtar']} {r['ad']:22s} [{r['tip']:11s}] ortak={len(r['ortak'])} "
        f"oran={r['oran']:.2f} karar={S.TRANSFER_KARAR[r['karar']]}")
yaz(f"\nOrtalama koruma oranı — sektör-içi={ici_ort:.2f} | sektör-ötesi={otesi_ort:.2f}")
yaz(f"Ortak kavram sayısı — sektör-içi={[len(r['ortak']) for r in ici]} | sektör-ötesi={[len(r['ortak']) for r in otesi]}")
gecen = [r["anahtar"] for r in sonuclar if r["karar"] == "dahil"]
yaz(f"Baraj geçen (DAHİL) senaryo: {gecen if gecen else 'YOK'}")
yaz("")
yaz(S.MSG6["bitti"])

_log = cfg.LOGS / "adim6_ozet.log"
_log.write_text("\n".join(CIKTI) + "\n", encoding="utf-8")
print(S.MSG["kayit"].format(yol=_log))

===== ÖZET — TRANSFER =====
  A1 Cell2Cell -> Iranian   [sektör-içi ] ortak=5 oran=0.18 karar=ZAYIF
  A2 Iranian -> Cell2Cell   [sektör-içi ] ortak=5 oran=0.69 karar=ZAYIF
  B1 telco -> bank          [sektör-ötesi] ortak=3 oran=0.53 karar=KISMÎ
  B2 bank -> telco          [sektör-ötesi] ortak=3 oran=0.33 karar=ZAYIF

Ortalama koruma oranı — sektör-içi=0.44 | sektör-ötesi=0.43
Ortak kavram sayısı — sektör-içi=[5, 5] | sektör-ötesi=[3, 3]
Baraj geçen (DAHİL) senaryo: YOK

ADIM 6 (transfer) tamamlandı. Yorum/karar kullanıcıya bırakıldı. Sağlamlık/yazım (Adım 7) yapılmadı.
Kaydedildi: /Users/emrahfidan/Desktop/churn-xai-profit/outputs/logs/adim6_ozet.log
